# Enhanced Federated Learning Cycle for DeepFake Detection

This notebook is **Flower-only** end-to-end.

Pipeline modules used:
- enhanced_client_selection.py
- update_validation.py
- knowledge_distillation.py
- client_reputation_ledger.py
- evaluation_metrics.py
- flwr_federated_cycle.py


In [ ]:
# 1) Install dependencies and import Flower pipeline modules
import subprocess
import sys

def _pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"pip install failed: {' '.join(packages)}")

_pip_install("flwr>=1.7")

from flwr_federated_cycle import (
    FLWRCycleConfig,
    FLWRFederatedLearningCycle,
    generate_proxy_data,
    generate_synthetic_data,
    partition_data_iid_flwr,
)

print("Flower environment is ready.")


## Configuration

Set experiment hyperparameters for the Flower federated cycle.


In [ ]:
config = FLWRCycleConfig(
    model_path="efficientnetb4_final.keras",
    num_devices=100,
    local_epochs=5,
    global_rounds=50,
    clients_per_round=15,
    local_batch_size=32,
    local_lr=1e-4,
    eval_every=10,
    enable_distillation=True,
    reports_dir="reports",
    tflite_output_path="effnet_global_flwr_final.tflite",
)

print("Config created:")
print(f"  rounds={config.global_rounds}, clients={config.num_devices}, per_round={config.clients_per_round}")


## Data Preparation (Real FF++ TFRecords)

Use this section when your TFRecords are already prepared.

Where to place your links and paths:
1. In the next cell, edit USER INPUTS.
2. Set GOOGLE_DRIVE_TFRECORD_FOLDER_URL only for Codespaces or when you want auto-download.
3. Set COLAB_DRIVE_TFRECORD_ROOT to your folder inside MyDrive.
4. Set KAGGLE_TFRECORD_ROOT to your Kaggle dataset mount path.
5. If running Codespaces with synced files, set CODESPACE_LOCAL_TFRECORD_ROOT.

In [ ]:
# 2) Load real FF++ TFRecords from Google Drive / Kaggle / Codespaces
import os
import sys
import glob
import re
import subprocess
from pathlib import Path

import tensorflow as tf


# ========================= USER INPUTS =========================
# Put your shared Google Drive folder link here if you want auto-download.
# Example: https://drive.google.com/drive/folders/1AbCdEf...xyz
GOOGLE_DRIVE_TFRECORD_FOLDER_URL = ""

# Colab: folder path after mounting MyDrive
COLAB_DRIVE_TFRECORD_ROOT = "/content/drive/MyDrive/ffpp_tfrecord_clients"

# Kaggle: dataset mount path (after Add data)
KAGGLE_TFRECORD_ROOT = "/kaggle/input/ff-c23-tfrecord/ffpp_tfrecord_clients"

# Codespaces/local fallback if files are already present in workspace
CODESPACE_LOCAL_TFRECORD_ROOT = "./ffpp_tfrecord_clients"

# Parsing and split config
IMG_SIZE = tuple(config.input_shape[:2])
TFRECORD_GLOB = "client_*.tfrecord"
COMPRESSION_TYPE = "GZIP"
VAL_CLIENTS = 10
TEST_CLIENTS = 10
SHUFFLE_BUFFER = 2048


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def _in_kaggle() -> bool:
    return os.path.isdir("/kaggle/input")


def _ensure_codespace_data(target_root: str) -> None:
    if os.path.isdir(target_root):
        return
    if not GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip():
        raise FileNotFoundError(
            "Codespaces/local path not found and GOOGLE_DRIVE_TFRECORD_FOLDER_URL is empty. "
            "Set one of them in USER INPUTS."
        )

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.0"])
    os.makedirs(target_root, exist_ok=True)

    subprocess.check_call([
        "gdown",
        "--folder",
        GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip(),
        "-O",
        target_root,
    ])


def resolve_tfrecord_root() -> str:
    if _in_colab():
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
        root = COLAB_DRIVE_TFRECORD_ROOT
    elif _in_kaggle():
        root = KAGGLE_TFRECORD_ROOT
    else:
        root = CODESPACE_LOCAL_TFRECORD_ROOT
        _ensure_codespace_data(root)

    if not os.path.isdir(root):
        raise FileNotFoundError(f"TFRecord root folder does not exist: {root}")
    return root


def parse_example(example_proto: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.float32),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.cast(parsed["label"], tf.float32)
    return image, label


def load_client_dataset(tfrecord_path: str) -> tf.data.Dataset:
    ds = tf.data.TFRecordDataset(
        tfrecord_path,
        compression_type=COMPRESSION_TYPE,
        num_parallel_reads=tf.data.AUTOTUNE,
    )
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    return ds


def extract_client_id(path: str) -> str:
    name = Path(path).name
    match = re.search(r"client_(\d+)", name)
    if match:
        return str(int(match.group(1)))
    return name.replace(".tfrecord", "")


def concat_datasets(dataset_list: list[tf.data.Dataset]) -> tf.data.Dataset:
    if not dataset_list:
        raise ValueError("No datasets provided for concatenation.")
    combined = dataset_list[0]
    for ds in dataset_list[1:]:
        combined = combined.concatenate(ds)
    return combined


root = resolve_tfrecord_root()
all_files = sorted(glob.glob(os.path.join(root, TFRECORD_GLOB)))

if len(all_files) < 3:
    raise RuntimeError(
        f"Expected multiple TFRecord client shards at {root}, found {len(all_files)}"
    )

# Split by client shard: train / val / test
n_total = len(all_files)
n_val = min(VAL_CLIENTS, max(1, n_total // 10))
n_test = min(TEST_CLIENTS, max(1, n_total // 10))

val_files = all_files[:n_val]
test_files = all_files[n_val:n_val + n_test]
train_files = all_files[n_val + n_test:]

if not train_files:
    raise RuntimeError("No train client TFRecords left after val/test split. Reduce VAL_CLIENTS/TEST_CLIENTS.")

# Build federated client datasets
selected_train_files = train_files[: config.num_devices]
client_data = {
    extract_client_id(fp): load_client_dataset(fp).shuffle(SHUFFLE_BUFFER, seed=42)
    for fp in selected_train_files
}

config.num_devices = len(client_data)

# Build server validation and test datasets
server_val_data = concat_datasets([load_client_dataset(fp) for fp in val_files])
test_data = concat_datasets([load_client_dataset(fp) for fp in test_files])

# Proxy data for KD: unlabeled stream from train clients
proxy_data = concat_datasets([client_data[cid].map(lambda pair: pair[0]) for cid in client_data])

print(f"TFRecord root: {root}")
print(f"Total client shards: {len(all_files)}")
print(f"Train/Val/Test shards: {len(selected_train_files)}/{len(val_files)}/{len(test_files)}")
print(f"Using num_devices={config.num_devices}")


## Build And Run Flower Cycle


In [ ]:
# 3) Build cycle and initialize model + modules
cycle = FLWRFederatedLearningCycle(config)
cycle.load_global_model()
cycle.create_clients(client_data)
cycle.setup_enhancement_modules()

print("Cycle initialized and ready to train.")


In [ ]:
# 4) Run Flower federated learning
history = cycle.run(
    server_val_data=server_val_data,
    test_data=test_data,
    proxy_data=proxy_data,
)

print("Training complete.")
print(f"Rounds completed: {len(history.get('round', []))}")


In [ ]:
# 5) Quick summary
if history.get("enhanced_accuracy"):
    best_acc = max(history["enhanced_accuracy"])
    final_acc = history["enhanced_accuracy"][-1]
    print(f"Best enhanced accuracy:  {best_acc:.4f}")
    print(f"Final enhanced accuracy: {final_acc:.4f}")

print(f"TFLite output: {config.tflite_output_path}")
print("Reports dir:  reports/")
